# CNN Hyperparameter Search

Train, test and visualize all search methods (manual, random, GA, PSO, ACO, Harmony Search) on every supported dataset (FashionMNIST, CIFAR10, CIFAR100).

## 1. Setup repo i środowiska

In [2]:
REPO_URL = "https://github.com/tsmyda/cnn-metaheuristics.git"
PROJECT_DIR = "/content/cnn-metaheuristics"

%cd /content
!rm -rf {PROJECT_DIR}
!git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt

/content
Cloning into '/content/cnn-metaheuristics'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 86 (delta 40), reused 71 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 193.33 KiB | 1.56 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/cnn-metaheuristics


## 2. Importy

In [3]:
import os
import sys
import importlib

sys.path.insert(0, "/content/cnn-metaheuristics")

import torch
import pandas as pd
import matplotlib.pyplot as plt

import src.utils
import src.search_space
import src.evaluator
import src.plots
import src.report_tables
import src.algorithms.manual_search
import src.algorithms.random_search
import src.algorithms.ga
import src.algorithms.pso
import src.algorithms.aco
import src.algorithms.harmony_search

for mod in [
    src.utils,
    src.search_space,
    src.evaluator,
    src.plots,
    src.report_tables,
    src.algorithms.manual_search,
    src.algorithms.random_search,
    src.algorithms.ga,
    src.algorithms.pso,
    src.algorithms.aco,
    src.algorithms.harmony_search,
]:
    importlib.reload(mod)

from src.utils import set_seed, ensure_dir
from src.algorithms.manual_search import run_manual_search
from src.algorithms.random_search import run_random_search
from src.algorithms.ga import run_ga
from src.algorithms.pso import run_pso
from src.algorithms.aco import run_aco
from src.algorithms.harmony_search import run_harmony_search
from src.plots import (
    plot_best_so_far,
    plot_time_to_best,
    plot_hyperparam_metric_correlation_heatmaps_by_method,
    save_summary_table,
)
from src.report_tables import (
    save_method_summary,
    save_best_configs,
    save_time_to_best,
)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 3. Parametry eksperymentu

In [4]:
DATASETS = ["FashionMNIST", "CIFAR10", "CIFAR100"]

METHODS = [
    "manual_search",
    "random_search",
    "ga",
    "pso",
    "aco",
    "harmony_search",
]

SEED = 42
EPOCHS = 5

RANDOM_BUDGET = 20

GA_POPULATION = 5
GA_GENERATIONS = 4
GA_MUTATION_PROB = 0.20
GA_ELITE_SIZE = 1
GA_TOURNAMENT_SIZE = 3

PSO_SWARM = 5
PSO_ITERATIONS = 4
PSO_W = 0.7
PSO_C1 = 1.5
PSO_C2 = 1.5

ACO_ANTS = 5
ACO_ITERATIONS = 4
ACO_EVAPORATION = 0.2
ACO_Q = 1.0
ACO_TOP_K = 2

HS_MEMORY_SIZE = 5
HS_ITERATIONS = 15
HS_HMCR = 0.9
HS_PAR = 0.3

assert RANDOM_BUDGET == GA_POPULATION * GA_GENERATIONS, "GA budget should match random search budget"
assert RANDOM_BUDGET == PSO_SWARM * PSO_ITERATIONS, "PSO budget should match random search budget"
assert RANDOM_BUDGET == ACO_ANTS * ACO_ITERATIONS, "ACO budget should match random search budget"
assert RANDOM_BUDGET == HS_MEMORY_SIZE + HS_ITERATIONS, "HS budget should match random search budget"

RESULTS_DIR = "results"
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")

ensure_dir(RESULTS_DIR)
ensure_dir(TABLES_DIR)
ensure_dir(FIGURES_DIR)

print("Datasets:", DATASETS)
print("Methods:", METHODS)
print("Seed:", SEED)
print("Epochs per evaluation:", EPOCHS)
print("Random budget:", RANDOM_BUDGET)
print("GA budget:", GA_POPULATION * GA_GENERATIONS)
print("PSO budget:", PSO_SWARM * PSO_ITERATIONS)
print("ACO budget:", ACO_ANTS * ACO_ITERATIONS)
print("HS budget:", HS_MEMORY_SIZE + HS_ITERATIONS)

Datasets: ['FashionMNIST', 'CIFAR10', 'CIFAR100']
Methods: ['manual_search', 'random_search', 'ga', 'pso', 'aco', 'harmony_search']
Seed: 42
Epochs per evaluation: 5
Random budget: 20
GA budget: 20
PSO budget: 20
ACO budget: 20
HS budget: 20


## 4. Sanity check losowania

In [5]:
from src.search_space import sample_config

for i in range(3):
    print(f"sample {i+1}:", sample_config())

sample 1: {'learning_rate': 0.0019004375238737116, 'batch_size': 32, 'num_blocks': 3, 'filters_1': 32, 'filters_2': 32, 'filters_3': 64, 'kernel_size': 3, 'dropout': 0.3682356070820062, 'dense_units': 256, 'optimizer': 'adamw', 'weight_decay': 0.0004748306046370143, 'use_batch_norm': 0}
sample 2: {'learning_rate': 0.0015169980582701848, 'batch_size': 32, 'num_blocks': 1, 'filters_1': 16, 'filters_2': 32, 'filters_3': 64, 'kernel_size': 3, 'dropout': 0.2806225314693065, 'dense_units': 256, 'optimizer': 'adamw', 'weight_decay': 0.0001270500735408539, 'use_batch_norm': 1}
sample 3: {'learning_rate': 0.00027598230915285764, 'batch_size': 128, 'num_blocks': 1, 'filters_1': 16, 'filters_2': 128, 'filters_3': 128, 'kernel_size': 5, 'dropout': 0.13893567083582092, 'dense_units': 64, 'optimizer': 'sgd', 'weight_decay': 2.0259598275933324e-06, 'use_batch_norm': 1}


## 5. Pomocnicza funkcja: uruchom wszystkie metody na jednym zbiorze

In [6]:
def run_all_methods_for_dataset(dataset_name: str) -> pd.DataFrame:
    """Run every method in METHODS on a single dataset and return a combined DataFrame."""
    dfs = []

    if "manual_search" in METHODS:
        print(f"\n### [{dataset_name}] Manual search")
        _, df_manual = run_manual_search(
            dataset_name=dataset_name,
            epochs=EPOCHS,
            device=device,
            seed=SEED,
        )
        dfs.append(df_manual)

    if "random_search" in METHODS:
        print(f"\n### [{dataset_name}] Random search")
        _, df_random = run_random_search(
            dataset_name=dataset_name,
            budget=RANDOM_BUDGET,
            epochs=EPOCHS,
            device=device,
            seed=SEED,
        )
        dfs.append(df_random)

    if "ga" in METHODS:
        print(f"\n### [{dataset_name}] Genetic Algorithm")
        _, df_ga = run_ga(
            dataset_name=dataset_name,
            population_size=GA_POPULATION,
            generations=GA_GENERATIONS,
            epochs=EPOCHS,
            device=device,
            seed=SEED,
            mutation_prob=GA_MUTATION_PROB,
            elite_size=GA_ELITE_SIZE,
            tournament_size=GA_TOURNAMENT_SIZE,
        )
        dfs.append(df_ga)

    if "pso" in METHODS:
        print(f"\n### [{dataset_name}] PSO")
        _, df_pso = run_pso(
            dataset_name=dataset_name,
            swarm_size=PSO_SWARM,
            iterations=PSO_ITERATIONS,
            epochs=EPOCHS,
            device=device,
            seed=SEED,
            w=PSO_W,
            c1=PSO_C1,
            c2=PSO_C2,
        )
        dfs.append(df_pso)

    if "aco" in METHODS:
        print(f"\n### [{dataset_name}] ACO")
        _, df_aco = run_aco(
            dataset_name=dataset_name,
            ants=ACO_ANTS,
            iterations=ACO_ITERATIONS,
            epochs=EPOCHS,
            device=device,
            seed=SEED,
            evaporation_rate=ACO_EVAPORATION,
            q=ACO_Q,
            top_k_deposit=ACO_TOP_K,
        )
        dfs.append(df_aco)

    if "harmony_search" in METHODS:
        print(f"\n### [{dataset_name}] Harmony Search")
        _, df_hs = run_harmony_search(
            dataset_name=dataset_name,
            harmony_memory_size=HS_MEMORY_SIZE,
            iterations=HS_ITERATIONS,
            epochs=EPOCHS,
            device=device,
            seed=SEED,
            hmcr=HS_HMCR,
            par=HS_PAR,
        )
        dfs.append(df_hs)

    df_all = pd.concat(dfs, ignore_index=True)
    df_all["dataset"] = dataset_name
    df_all["seed"] = SEED
    return df_all

## 6. Główna pętla: wszystkie metody x wszystkie zbiory

Uwaga: ten krok jest kosztowny. Dla każdego zbioru uruchamiamy 6 metod po ~20 ewaluacji.

In [ ]:
results_per_dataset: dict[str, pd.DataFrame] = {}

for dataset_name in DATASETS:
    print(f"\n================ Dataset: {dataset_name} ================")
    set_seed(SEED)
    df_dataset = run_all_methods_for_dataset(dataset_name)

    dataset_csv = os.path.join(TABLES_DIR, f"all_methods_{dataset_name}.csv")
    df_dataset.to_csv(dataset_csv, index=False)
    results_per_dataset[dataset_name] = df_dataset
    print(f"Saved {dataset_name} results to {dataset_csv}")

df_all = pd.concat(results_per_dataset.values(), ignore_index=True)
all_csv = os.path.join(TABLES_DIR, "all_methods_all_datasets.csv")
df_all.to_csv(all_csv, index=False)
print("\nSaved combined results to", all_csv)
df_all[["dataset", "method", "iteration", "val_accuracy", "test_accuracy", "time_sec"]].head()


================ Dataset: FashionMNIST ================

### [FashionMNIST] Manual search


100%|██████████| 26.4M/26.4M [00:02<00:00, 13.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 204kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.81MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 12.8MB/s]


[MANUAL] iter=01/5 | val_acc=0.9177 | time=59.2s
[MANUAL] iter=02/5 | val_acc=0.9033 | time=40.8s
[MANUAL] iter=03/5 | val_acc=0.9088 | time=53.4s
[MANUAL] iter=04/5 | val_acc=0.8850 | time=54.3s
[MANUAL] iter=05/5 | val_acc=0.9228 | time=66.1s

### [FashionMNIST] Random search
[RS] iter=01/20 | val_acc=0.8952 | time=76.6s
[RS] iter=02/20 | val_acc=0.9138 | time=65.9s
[RS] iter=03/20 | val_acc=0.8817 | time=39.8s
[RS] iter=04/20 | val_acc=0.8748 | time=71.8s
[RS] iter=05/20 | val_acc=0.9142 | time=66.1s
[RS] iter=06/20 | val_acc=0.8912 | time=40.3s
[RS] iter=07/20 | val_acc=0.9093 | time=94.7s
[RS] iter=08/20 | val_acc=0.9223 | time=66.6s
[RS] iter=09/20 | val_acc=0.8950 | time=98.3s
[RS] iter=10/20 | val_acc=0.1022 | time=54.1s
[RS] iter=11/20 | val_acc=0.8905 | time=143.3s
[RS] iter=12/20 | val_acc=0.1097 | time=54.0s
[RS] iter=13/20 | val_acc=0.9107 | time=35.3s
[RS] iter=14/20 | val_acc=0.9187 | time=50.6s
[RS] iter=15/20 | val_acc=0.8855 | time=60.6s
[RS] iter=16/20 | val_acc=0.92

100%|██████████| 170M/170M [00:04<00:00, 40.3MB/s] 


[MANUAL] iter=01/5 | val_acc=0.6780 | time=63.6s
[MANUAL] iter=02/5 | val_acc=0.6452 | time=46.6s
[MANUAL] iter=03/5 | val_acc=0.6536 | time=55.5s
[MANUAL] iter=04/5 | val_acc=0.6256 | time=54.4s
[MANUAL] iter=05/5 | val_acc=0.6412 | time=66.4s

### [CIFAR10] Random search
[RS] iter=01/20 | val_acc=0.5580 | time=75.0s
[RS] iter=02/20 | val_acc=0.6416 | time=64.6s
[RS] iter=03/20 | val_acc=0.5384 | time=42.0s
[RS] iter=04/20 | val_acc=0.5434 | time=76.4s
[RS] iter=05/20 | val_acc=0.5902 | time=64.8s
[RS] iter=06/20 | val_acc=0.5762 | time=43.5s
[RS] iter=07/20 | val_acc=0.5844 | time=106.9s
[RS] iter=08/20 | val_acc=0.6888 | time=65.8s
[RS] iter=09/20 | val_acc=0.6382 | time=115.4s
[RS] iter=10/20 | val_acc=0.1064 | time=54.8s
[RS] iter=11/20 | val_acc=0.1024 | time=155.5s
[RS] iter=12/20 | val_acc=0.4004 | time=54.9s
[RS] iter=13/20 | val_acc=0.6356 | time=39.1s
[RS] iter=14/20 | val_acc=0.1064 | time=50.9s
[RS] iter=15/20 | val_acc=0.5726 | time=72.9s
[RS] iter=16/20 | val_acc=0.6202 

## 7. Wykresy best-so-far per zbiór danych

In [ ]:
best_so_far_paths = {}

for dataset_name, df_dataset in results_per_dataset.items():
    dataset_csv = os.path.join(TABLES_DIR, f"all_methods_{dataset_name}.csv")
    fig_path = os.path.join(FIGURES_DIR, f"best_so_far_{dataset_name}.png")
    plot_best_so_far(dataset_csv, fig_path)
    best_so_far_paths[dataset_name] = fig_path
    print("Saved:", fig_path)

fig, axes = plt.subplots(1, len(DATASETS), figsize=(6 * len(DATASETS), 5))
if len(DATASETS) == 1:
    axes = [axes]
for ax, dataset_name in zip(axes, DATASETS):
    img = plt.imread(best_so_far_paths[dataset_name])
    ax.imshow(img)
    ax.set_title(dataset_name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Wykresy time-to-best per zbiór danych

In [ ]:
time_to_best_per_dataset = {}

for dataset_name, df_dataset in results_per_dataset.items():
    ttb_csv = os.path.join(TABLES_DIR, f"time_to_best_{dataset_name}.csv")
    ttb_df = save_time_to_best(df_dataset, ttb_csv)
    time_to_best_per_dataset[dataset_name] = ttb_df

    fig_path = os.path.join(FIGURES_DIR, f"time_to_best_{dataset_name}.png")
    plot_time_to_best(ttb_df, fig_path)
    print("Saved:", fig_path)

pd.concat(
    [df.assign(dataset=name) for name, df in time_to_best_per_dataset.items()],
    ignore_index=True,
)

## 9. Hyperparameter correlation heatmaps (per zbiór danych, per metoda)

In [ ]:
for dataset_name, df_dataset in results_per_dataset.items():
    out_dir = os.path.join(FIGURES_DIR, f"hyperparam_correlations_{dataset_name}")
    plot_hyperparam_metric_correlation_heatmaps_by_method(df_dataset, out_dir)
    print("Saved heatmaps for", dataset_name, "to", out_dir)

## 10. Tabele podsumowujące (per zbiór danych)

In [ ]:
summaries = {}
for dataset_name, df_dataset in results_per_dataset.items():
    summary_path = os.path.join(TABLES_DIR, f"method_summary_{dataset_name}.csv")
    summary = save_method_summary(df_dataset, summary_path)
    summaries[dataset_name] = summary
    print(f"\n=== {dataset_name} ===")
    display(summary)

## 11. Najlepsze konfiguracje na każdy zbiór danych x metodę

In [ ]:
best_configs_per_dataset = {}
for dataset_name, df_dataset in results_per_dataset.items():
    best_path = os.path.join(TABLES_DIR, f"best_configs_{dataset_name}.csv")
    best_df = save_best_configs(df_dataset, best_path)
    best_configs_per_dataset[dataset_name] = best_df
    print(f"\n=== Best per method on {dataset_name} ===")
    display(best_df[[
        "method",
        "iteration",
        "learning_rate",
        "batch_size",
        "num_blocks",
        "filters_1",
        "filters_2",
        "filters_3",
        "kernel_size",
        "dropout",
        "dense_units",
        "val_accuracy",
        "test_accuracy",
        "time_sec",
        "num_params",
    ]])

## 12. Wykres jakość vs skumulowany czas

In [ ]:
def plot_best_so_far_vs_time(df: pd.DataFrame, output_path: str, title: str) -> None:
    plt.figure(figsize=(8, 5))
    for method, group in df.groupby("method"):
        group = group.sort_values("iteration").copy()
        group["cum_time_sec"] = group["time_sec"].cumsum()
        group["best_so_far"] = group["val_accuracy"].cummax()
        plt.plot(group["cum_time_sec"], group["best_so_far"], label=method)
    plt.xlabel("Skumulowany czas [s]")
    plt.ylabel("Best validation accuracy")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()

for dataset_name, df_dataset in results_per_dataset.items():
    fig_path = os.path.join(FIGURES_DIR, f"best_so_far_vs_time_{dataset_name}.png")
    plot_best_so_far_vs_time(df_dataset, fig_path, f"Quality vs cumulative time - {dataset_name}")
    print("Saved:", fig_path)

## 13. Cross-dataset: najlepsza val_accuracy per metoda x zbiór

In [ ]:
cross = (
    df_all.groupby(["dataset", "method"])["val_accuracy"].max().unstack("dataset").round(4)
)
cross_path = os.path.join(TABLES_DIR, "cross_dataset_best_val_accuracy.csv")
cross.to_csv(cross_path)
print("Saved:", cross_path)

fig, ax = plt.subplots(figsize=(1.5 * len(METHODS) + 3, 4))
cross.plot(kind="bar", ax=ax)
ax.set_ylabel("Best val_accuracy")
ax.set_title("Najlepsza val_accuracy - metoda x zbiór danych")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
cross_fig_path = os.path.join(FIGURES_DIR, "cross_dataset_best_val_accuracy.png")
plt.savefig(cross_fig_path, dpi=200)
plt.show()
print("Saved:", cross_fig_path)
cross

## 14. Final retraining najlepszych konfiguracji

Dla każdej pary (zbiór danych, metoda) bierzemy najlepszą konfigurację ze stage'u wyszukiwania i dotrenowujemy ją na większej liczbie epok.

In [ ]:
FINAL_EPOCHS = 20
FINAL_SEED = 42

print("Final retraining epochs:", FINAL_EPOCHS)

In [ ]:
from src.datasets import get_dataset_loaders
from src.model import TunableCNN
from src.train import train_one_epoch, evaluate
from src.utils import count_parameters
from torch.optim import Adam


def retrain_best_config(
    config: dict,
    dataset_name: str,
    epochs: int,
    device: str,
    seed: int = 42,
    val_split: float = 0.1,
    num_workers: int = 2,
):
    import numpy as np
    import torch

    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    train_loader, val_loader, test_loader, image_channels, image_size, num_classes = get_dataset_loaders(
        dataset_name=dataset_name,
        batch_size=int(config["batch_size"]),
        val_split=val_split,
        num_workers=num_workers,
        seed=seed,
    )

    model = TunableCNN(
        image_channels=image_channels,
        image_size=image_size,
        num_classes=num_classes,
        num_blocks=int(config["num_blocks"]),
        filters_1=int(config["filters_1"]),
        filters_2=int(config["filters_2"]),
        filters_3=int(config["filters_3"]),
        kernel_size=int(config["kernel_size"]),
        dropout=float(config["dropout"]),
        dense_units=int(config["dense_units"]),
    ).to(device)

    optimizer = Adam(model.parameters(), lr=float(config["learning_rate"]))

    history = []
    best_val_acc = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, device)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc = evaluate(model, test_loader, device)

    return {
        "best_val_acc_retrain": best_val_acc,
        "final_test_acc": test_acc,
        "final_test_loss": test_loss,
        "num_params": count_parameters(model),
        "history": pd.DataFrame(history),
    }

In [ ]:
final_results = []
history_frames = []

for dataset_name, best_df in best_configs_per_dataset.items():
    for _, row in best_df.iterrows():
        method = row["method"]
        config = {
            "learning_rate": row["learning_rate"],
            "batch_size": row["batch_size"],
            "num_blocks": row["num_blocks"],
            "filters_1": row["filters_1"],
            "filters_2": row["filters_2"],
            "filters_3": row["filters_3"],
            "kernel_size": row["kernel_size"],
            "dropout": row["dropout"],
            "dense_units": row["dense_units"],
        }

        print(f"\n=== Final retraining: {dataset_name} / {method} ===")
        print(config)

        out = retrain_best_config(
            config=config,
            dataset_name=dataset_name,
            epochs=FINAL_EPOCHS,
            device=device,
            seed=FINAL_SEED,
        )

        final_results.append({
            "dataset": dataset_name,
            "method": method,
            "search_best_val_accuracy": row["val_accuracy"],
            "search_test_accuracy": row["test_accuracy"],
            "retrain_best_val_accuracy": out["best_val_acc_retrain"],
            "final_test_accuracy": out["final_test_acc"],
            "final_test_loss": out["final_test_loss"],
            "num_params": out["num_params"],
        })

        hist = out["history"].copy()
        hist["dataset"] = dataset_name
        hist["method"] = method
        history_frames.append(hist)

final_results_df = pd.DataFrame(final_results).sort_values(
    ["dataset", "final_test_accuracy"], ascending=[True, False]
)
final_history_df = pd.concat(history_frames, ignore_index=True)

final_results_df

## 15. Zapis wyników finalnego retrainingu

In [ ]:
final_results_path = os.path.join(TABLES_DIR, "final_retraining_results.csv")
final_history_path = os.path.join(TABLES_DIR, "final_retraining_history.csv")

final_results_df.to_csv(final_results_path, index=False)
final_history_df.to_csv(final_history_path, index=False)

print("Saved final retraining results to:", final_results_path)
print("Saved final retraining history to:", final_history_path)

## 16. Wykresy finalnego treningu (per zbiór danych)

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(6 * len(DATASETS), 5), sharey=False)
if len(DATASETS) == 1:
    axes = [axes]

for ax, dataset_name in zip(axes, DATASETS):
    subset = final_history_df[final_history_df["dataset"] == dataset_name]
    for method, group in subset.groupby("method"):
        ax.plot(group["epoch"], group["val_acc"], label=method)
    ax.set_title(f"Final retraining - {dataset_name}")
    ax.set_xlabel("Epoka")
    ax.set_ylabel("Validation accuracy")
    ax.legend()

plt.tight_layout()
final_curves_path = os.path.join(FIGURES_DIR, "final_retraining_val_curves.png")
plt.savefig(final_curves_path, dpi=200)
plt.show()
print("Saved:", final_curves_path)

## 17. Final test accuracy - metoda x zbiór danych

In [ ]:
final_pivot = (
    final_results_df.pivot(index="method", columns="dataset", values="final_test_accuracy").round(4)
)
final_pivot_path = os.path.join(TABLES_DIR, "final_test_accuracy_pivot.csv")
final_pivot.to_csv(final_pivot_path)

fig, ax = plt.subplots(figsize=(1.5 * len(METHODS) + 3, 4))
final_pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("Final test accuracy")
ax.set_title("Final test accuracy po retrainingu - metoda x zbiór")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
final_pivot_fig_path = os.path.join(FIGURES_DIR, "final_test_accuracy_pivot.png")
plt.savefig(final_pivot_fig_path, dpi=200)
plt.show()
print("Saved:", final_pivot_path)
print("Saved:", final_pivot_fig_path)
final_pivot

## 18. Końcowa tabela do raportu

In [ ]:
analysis_rows = []
for dataset_name, df_dataset in results_per_dataset.items():
    agg = (
        df_dataset.groupby("method", as_index=False)
        .agg(
            best_val_accuracy=("val_accuracy", "max"),
            mean_val_accuracy=("val_accuracy", "mean"),
            best_test_accuracy=("test_accuracy", "max"),
            mean_time_sec=("time_sec", "mean"),
            total_time_sec=("time_sec", "sum"),
            mean_num_params=("num_params", "mean"),
        )
    )
    agg["dataset"] = dataset_name
    analysis_rows.append(agg)

analysis_df = pd.concat(analysis_rows, ignore_index=True)

report_table = final_results_df.merge(
    analysis_df[["dataset", "method", "total_time_sec"]],
    on=["dataset", "method"],
    how="left",
)

report_table = report_table[[
    "dataset",
    "method",
    "search_best_val_accuracy",
    "retrain_best_val_accuracy",
    "final_test_accuracy",
    "total_time_sec",
    "num_params",
]].sort_values(["dataset", "final_test_accuracy"], ascending=[True, False])

report_path = os.path.join(TABLES_DIR, "final_report_table.csv")
report_table.to_csv(report_path, index=False)
print("Saved:", report_path)
report_table

## 19. (Opcjonalnie) Zapis wyników na Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/cnn-metaheuristics-results
# !cp -r {RESULTS_DIR} /content/drive/MyDrive/cnn-metaheuristics-results/